In [1]:
import os

os.environ["ACCELERATE_TORCH_DEVICE"] = "cpu"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0.0" 

In [2]:
from transformers import pipeline

In [3]:
classificador_sentimento = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [4]:
classificador_sentimento('I love this product!')

[{'label': 'POSITIVE', 'score': 0.9998855590820312}]

In [5]:
classificador_sentimento('I am tottaly disappointed with this product! It is not worth the money I spent on it.')

[{'label': 'NEGATIVE', 'score': 0.9998266100883484}]

## Selecionando um modelo para analise de sentimento em PT

In [6]:
%pip install pysentimiento 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/devbodegami/Projetos/formacao-especialista-em-ia/.venv-test/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
from pysentimiento import create_analyzer

In [8]:
modelo_analise_sentimento = create_analyzer(task="sentiment", lang="pt", device="cpu")

In [9]:
modelo_analise_sentimento.predict('''
A fritadeira é sensacional, muito além do que eu imaginava. É linda, super funcional e muito fácil de manusear.
Fácil de limpar e potente. Super recomendo!
''')

AnalyzerOutput(output=POS, probas={POS: 0.989, NEU: 0.009, NEG: 0.002})

In [10]:
modelo_analise_sentimento.predict('''
Após pouco meses de uso a carcaça de aço escovado começou a oxidar, demonstrando a baixa qualidade de proteção.
Fora esse detalhe, o produto cumpre o prometido. 
''')

AnalyzerOutput(output=NEU, probas={NEU: 0.901, NEG: 0.086, POS: 0.013})

In [11]:
modelo_analise_sentimento.predict('''
Em menos de 1 ano parou de funcionar, enviei para assistência técnica por estar na garantia,
trocaram o motor, passou a ficar menos potente e não durou 2 utilizações.
Isso se repetiu várias vezes, até que desisti de ficar levando lá e queimando de novo em seguida,
vi outros clientes com o mesmo problema. Não comprem!
''')

AnalyzerOutput(output=NEG, probas={NEG: 0.911, NEU: 0.072, POS: 0.017})

## Aplicando o modelo aos Dados

In [12]:
import pandas as pd

In [13]:
dados = pd.read_csv('dados/resenhas.csv')

In [14]:
dados[:5]

,ID,Resenha
0,24,"A fritadeira é sensacional, muito além do que ..."
1,733,"Após usar o produto, achei-o fácil e muito efi..."
2,865,"Muito funcional, prática e moderna."
3,809,"Boa, mas não das melhores pois a frente de vid..."
4,628,Eu comecei a usar e é bem espaçosa. Gostei poi...


In [15]:
# import os

# os.environ["ACCELERATE_TORCH_DEVICE"] = "cpu"
# os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0.0" 

In [16]:
resultados_previsao = modelo_analise_sentimento.predict(dados['Resenha'])

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Map:   0%|          | 0/36 [00:00<?, ? examples/s]

In [19]:
resultados_previsao[:5]

[AnalyzerOutput(output=POS, probas={POS: 0.989, NEU: 0.009, NEG: 0.002}),
 AnalyzerOutput(output=POS, probas={POS: 0.670, NEU: 0.320, NEG: 0.011}),
 AnalyzerOutput(output=POS, probas={POS: 0.900, NEU: 0.097, NEG: 0.003}),
 AnalyzerOutput(output=NEG, probas={NEG: 0.731, NEU: 0.259, POS: 0.010}),
 AnalyzerOutput(output=POS, probas={POS: 0.976, NEU: 0.021, NEG: 0.003})]

In [20]:
sentimento = []

for resultado in resultados_previsao:
    sentimento.append(resultado.output)

In [22]:
sentimento[:5]

['POS', 'POS', 'POS', 'NEG', 'POS']

In [23]:
dados["Sentimento"] = sentimento

In [24]:
dados[:5]

,ID,Resenha,Sentimento
0,24,"A fritadeira é sensacional, muito além do que ...",POS
1,733,"Após usar o produto, achei-o fácil e muito efi...",POS
2,865,"Muito funcional, prática e moderna.",POS
3,809,"Boa, mas não das melhores pois a frente de vid...",NEG
4,628,Eu comecei a usar e é bem espaçosa. Gostei poi...,POS
